# A1.5 — Le split figé

On sépare les molécules (`inchikey14`), et non les spectres : tous les spectres d'une molécule sont du même côté. Une molécule de validation n'a donc aucun spectre dans l'entraînement, comme les molécules du test.

Le split est produit par `python -m casmi.split` (code dans `src/casmi/split.py`) et lu avec `load_split()`.

In [ ]:
%load_ext autoreload
%autoreload 2

## Charger le split et le relier aux spectres

In [ ]:
import pandas as pd
from casmi.io import load_meta
from casmi.split import load_split

split = load_split()
meta = load_meta(["inchikey14", "ingest_lib", "adduct", "ionization_mode"])
df = meta.merge(split, on="inchikey14", validate="many_to_one")
split.head()

## Nombre de molécules et de spectres dans chaque partie

In [ ]:
sizes = df.groupby("split").agg(molecules=("inchikey14", "nunique"), spectres=("row_id", "size"))
sizes.loc["dont dev"] = [df.loc[df["dev"], "inchikey14"].nunique(), df["dev"].sum()]
sizes

## Nombre de spectres par molécule

In [ ]:
df.groupby(["split", "inchikey14"]).size().groupby("split").describe(percentiles=[0.5, 0.9, 0.99]).round(1)

## Distributions comparées (% des spectres)

Si le tirage est équilibré, chaque bibliothèque, adduct et mode d'ionisation pèse à peu près autant des deux côtés.

In [ ]:
def compare(col, top=10):
    t = pd.crosstab(df[col], df["split"], normalize="columns").mul(100).round(1)
    t["écart"] = (t["val"] - t["train"]).round(1)
    return t.sort_values("train", ascending=False).head(top)

compare("ingest_lib")

In [ ]:
compare("adduct")

In [ ]:
compare("ionization_mode")